In [12]:
from pathlib import Path
import os
import math
import time
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.stats import pearsonr, spearmanr

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import normalize
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv


In [4]:
# =========================
# === Config: file paths ===
# =========================
PATHS = {
    "gpt4o": {
        "train": "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/Synthetic_Comorbidity_LLM/gpt-4o-mini/train/gpt4o_train_merged.csv",
        "test" : "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/Synthetic_Comorbidity_LLM/gpt-4o-mini/test/gpt4o_test_merged.csv",
    },
    "gemini": {
        "train": "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/Synthetic_Comorbidity_LLM/Gemini/train/gemini_train_merged.csv",
        "test" : "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/Synthetic_Comorbidity_LLM/Gemini/test/gemini_test_merged.csv",
    },
    "deepseek": {
        "train": "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/Synthetic_Comorbidity_LLM/Deepseek/train/deepseek_train_merged.csv",
        "test" : "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/Synthetic_Comorbidity_LLM/Deepseek/test/deepseek_test_merged.csv",
    },
}

WIDE_TRAIN = "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/train_50K_llm_synthetic_df_wide.csv"
WIDE_TEST  = "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/test_10K_llm_synthetic_df_wide.csv"


In [7]:
ART_DIR = Path("mlp_llm_targets_artifacts")
ART_DIR.mkdir(exist_ok=True, parents=True)

SEED = 42
rng = np.random.default_rng(SEED)

In [8]:
# =========================
# === Helpers ============
# =========================
def ensure_pid_str(df: pd.DataFrame) -> pd.DataFrame:
    if 'participant_id' in df.columns:
        df['participant_id'] = df['participant_id'].astype(str)
    return df

def load_llm_scores(path: str, model_name: str, split_name: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {model_name} {split_name} file: {path}")
    df = pd.read_csv(path)
    df = ensure_pid_str(df)
    if 'participant_id' not in df.columns or 'score' not in df.columns:
        raise ValueError(f"{model_name} {split_name} file must contain participant_id and score: {path}")
    return df[['participant_id', 'score']].copy()

def build_common_split(split: str) -> pd.DataFrame:
    """Return participant_id + score_gpt4o + score_gemini + score_deepseek + llm_mean_score."""
    gpt4o   = load_llm_scores(PATHS['gpt4o'][split],  'gpt4o',  split).rename(columns={'score':'score_gpt4o'})
    gemini  = load_llm_scores(PATHS['gemini'][split], 'gemini', split).rename(columns={'score':'score_gemini'})
    deepseek= load_llm_scores(PATHS['deepseek'][split],'deepseek',split).rename(columns={'score':'score_deepseek'})

    tmp  = gpt4o.merge(gemini,  on='participant_id', how='inner')
    all3 = tmp.merge(deepseek, on='participant_id', how='inner')

    score_cols = ['score_gpt4o', 'score_gemini', 'score_deepseek']
    all3 = all3.dropna(subset=score_cols)
    all3['llm_mean_score'] = all3[score_cols].mean(axis=1)

    print(f"[{split.upper()}] Common participants across GPT-4o, Gemini, DeepSeek: {len(all3)}")
    return all3[['participant_id'] + score_cols + ['llm_mean_score']].copy()


In [10]:
# =========================
# === Load & Merge ========
# =========================
train_df = pd.read_csv(WIDE_TRAIN)
test_df  = pd.read_csv(WIDE_TEST)

if 'participant_id' not in train_df.columns:
    train_df = train_df.reset_index().rename(columns={'index': 'participant_id'})
if 'participant_id' not in test_df.columns:
    test_df  = test_df.reset_index().rename(columns={'index': 'participant_id'})

train_df = ensure_pid_str(train_df)
test_df  = ensure_pid_str(test_df)

train_scores_all = build_common_split('train')
test_scores_all  = build_common_split('test')

train_merged = train_df.merge(train_scores_all, on='participant_id', how='inner')
test_merged  = test_df.merge(test_scores_all,  on='participant_id', how='inner')

print("Train merged shape:", train_merged.shape)
print("Test merged shape :", test_merged.shape)

# =========================
# === Features/Targets ====
# =========================
target_cols = ['score_gpt4o', 'score_gemini', 'score_deepseek', 'llm_mean_score']

# drop IDs + targets (+ any columns you choose not to use, e.g., ethnicity_name)
drop_cols = ['participant_id'] + target_cols + ['ethnicity_name']
X_train_raw = train_merged.drop(columns=[c for c in drop_cols if c in train_merged.columns])
X_test_raw  = test_merged.drop(columns=[c for c in drop_cols if c in test_merged.columns])

X_train = pd.get_dummies(X_train_raw, dummy_na=True)
X_test  = pd.get_dummies(X_test_raw,  dummy_na=True)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)


[TRAIN] Common participants across GPT-4o, Gemini, DeepSeek: 49210
[TEST] Common participants across GPT-4o, Gemini, DeepSeek: 9872
Train merged shape: (49210, 270)
Test merged shape : (9872, 270)
X_train shape: (49210, 264)
X_test  shape: (9872, 264)


In [16]:
# =================================================================
# === Torch MLP + Hyperparameter Search (per target) ==============
# =================================================================

import math, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import joblib

# artifacts dir (ensure it exists)
ART_DIR = Path("mlp_llm_targets_artifacts")
ART_DIR.mkdir(exist_ok=True, parents=True)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def make_activation(name: str):
    return {"relu": nn.ReLU, "leakyrelu": nn.LeakyReLU}[name]()

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, dropout, activation='relu'):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(last, h), make_activation(activation), nn.Dropout(dropout)]
            last = h
        layers += [nn.Linear(last, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def sample_arch(
    rng,
    min_layers=3,
    max_layers=6,
    width_choices=(128, 256, 512, 1024),
    pattern_choices=("constant", "pyramid", "free", "exp_decay")
):
    L = int(rng.integers(min_layers, max_layers + 1))
    pattern = rng.choice(pattern_choices)
    if hasattr(pattern, "item"):
        pattern = pattern.item()

    if pattern == "constant":
        w = int(rng.choice(width_choices))
        hidden = tuple([w] * L)
    elif pattern == "pyramid":
        widths = sorted([int(rng.choice(width_choices)) for _ in range(L)], reverse=True)
        hidden = tuple(widths)
    elif pattern == "exp_decay":
        w0 = int(rng.choice(width_choices))
        hidden = tuple(max(32, w0 // (2 ** i)) for i in range(L))
    else:  # "free"
        hidden = tuple(int(rng.choice(width_choices)) for _ in range(L))
    return hidden

def kfold_split_indices(n, val_frac=0.1, seed=SEED):
    rng_local = np.random.default_rng(seed)
    idx = np.arange(n)
    rng_local.shuffle(idx)
    n_val = max(1, int(val_frac * n)) if n > 10 else 1
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]
    return train_idx, val_idx

def train_eval_one(
    Xtr, ytr, Xva, yva, Xte, yte,
    in_dim,
    hidden_dims=(512,256,128),
    dropout=0.2,
    activation='relu',
    lr=1e-3,
    weight_decay=1e-5,
    batch_size=256,
    max_epochs=200,
    patience=12,
    clip_norm=1.0,
    verbose=False
):
    model = MLP(in_dim, hidden_dims, dropout, activation=activation).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
    )

    def make_loader(X, y, bs, shuffle):
        ds = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).float().reshape(-1,1))
        return DataLoader(ds, batch_size=bs, shuffle=shuffle)

    train_loader = make_loader(Xtr, ytr, batch_size, True)
    val_loader   = make_loader(Xva, yva, batch_size, False)
    test_loader  = make_loader(Xte, yte, batch_size, False)

    best_val = float("inf"); best_state=None; best_epoch=0; no_improve=0
    train_losses, val_losses, lrs = [], [], []

    for epoch in range(1, max_epochs+1):
        # ---- Train ----
        model.train(); tr_loss=0.0; n=0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            optimizer.step()
            tr_loss += loss.item() * xb.size(0); n += xb.size(0)
        tr_loss /= max(1,n)

        # ---- Val ----
        model.eval(); va_loss=0.0; n=0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                loss = criterion(model(xb), yb)
                va_loss += loss.item() * xb.size(0); n += xb.size(0)
        va_loss /= max(1,n)

        # scheduler + logs
        scheduler.step(va_loss)
        train_losses.append(tr_loss)
        val_losses.append(va_loss)
        lrs.append(optimizer.param_groups[0]['lr'])

        if verbose:
            print(f"  epoch {epoch:03d} | train={tr_loss:.6f} | val={va_loss:.6f} | lr={optimizer.param_groups[0]['lr']:.2e}")

        # early stopping tracking
        if va_loss + 1e-6 < best_val:
            best_val = va_loss; best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f"  early stop @ {epoch}, best {best_epoch}, val {best_val:.6f}")
                break

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state); model.to(DEVICE)

    # ---- Test ----
    model.eval(); preds=[]; ys=[]
    with torch.no_grad():
        for xb, yb in test_loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
            ys.append(yb.cpu().numpy())
    y_pred = np.vstack(preds).ravel()
    y_true = np.vstack(ys).ravel()

    mse  = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    pr, pp = pearsonr(y_true, y_pred)
    sr, sp = spearmanr(y_true, y_pred)

    return {
        "model": model,
        "metrics": {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2,
                    "pearson_r": pr, "pearson_p": pp, "spearman_r": sr, "spearman_p": sp},
        "best_val": best_val,
        "best_epoch": best_epoch,
        "y_true": y_true,
        "y_pred": y_pred,
        "logs": {  # <-- per-epoch loss curve from this training run
            "epoch": list(range(1, len(train_losses)+1)),
            "train_loss": train_losses,
            "val_loss": val_losses,
            "lr": lrs
        }
    }

# =======================
# === HPO per target ====
# =======================
MAX_TRIALS = 25
DROPOUTS   = [0.1, 0.2, 0.3, 0.4]
ACTS       = ('relu', 'leakyrelu')
LRS        = [1e-4, 3e-4, 1e-3]
WDS        = [0.0, 1e-6, 1e-5, 1e-4]
BATCHES    = [64, 128, 256, 512]

feat_list_global = list(X_train.columns)

# master RNG → derive a per-target RNG (not per-trial)
rng_master = np.random.default_rng(SEED)

for target in target_cols:
    print(f"\n==============================")
    print(f"Target: {target}")
    print(f"==============================")

    child_seed = int(rng_master.integers(0, 2**31 - 1))
    rng = np.random.default_rng(child_seed)

    y_train_vec = train_merged[target].astype(float).values
    y_test_vec  = test_merged[target].astype(float).values
    Xtr_np = X_train.values.astype("float32")
    Xte_np = X_test.values.astype("float32")

    tr_idx, va_idx = kfold_split_indices(len(Xtr_np), val_frac=0.1, seed=SEED)
    Xtr, ytr = Xtr_np[tr_idx], y_train_vec[tr_idx]
    Xva, yva = Xtr_np[va_idx], y_train_vec[va_idx]

    in_dim = Xtr.shape[1]
    best = None; best_conf = None; best_logs = None
    trial_rows = []

    for t in range(1, MAX_TRIALS+1):
        hidden   = sample_arch(rng, min_layers=3, max_layers=6)
        dropout  = float(rng.choice(DROPOUTS))
        act_raw  = rng.choice(ACTS)
        act      = act_raw.item() if hasattr(act_raw, "item") else act_raw
        act      = str(act)
        lr       = float(rng.choice(LRS))
        wd       = float(rng.choice(WDS))
        bs       = int(rng.choice(BATCHES))

        out = train_eval_one(
            Xtr, ytr, Xva, yva, Xte_np, y_test_vec,
            in_dim=in_dim,
            hidden_dims=hidden,
            dropout=dropout,
            activation=act,
            lr=lr,
            weight_decay=wd,
            batch_size=bs,
            max_epochs=200,
            patience=12,
            clip_norm=1.0,
            verbose=False
        )
        metrics = out["metrics"]

        tried_conf = {
            "hidden": hidden,
            "dropout": dropout,
            "activation": act,
            "lr": lr,
            "weight_decay": wd,
            "batch_size": bs
        }

        row = {
            "trial": t,
            "hidden": str(hidden),
            "dropout": dropout,
            "activation": act,
            "lr": lr,
            "weight_decay": wd,
            "batch_size": bs,
            "val_best": out["best_val"],
            "test_mse": metrics["mse"],
            "test_rmse": metrics["rmse"],
            "test_mae": metrics["mae"],
            "test_r2": metrics["r2"],
            "test_pearson_r": metrics["pearson_r"],
            "test_spearman_r": metrics["spearman_r"],
        }
        trial_rows.append(row)

        # update best-so-far (and keep its loss logs)
        if (best is None) or (out["best_val"] < best["best_val"]):
            best = out
            best_conf = tried_conf
            best_logs = out["logs"]

        print(
            f"  Trial {t:02d} | val*={out['best_val']:.6f} | "
            f"test RMSE={metrics['rmse']:.5f} | r={metrics['pearson_r']:.4f}\n"
            f"    Tried: {tried_conf}\n"
            f"    Best : {best_conf}\n"
        )

    # Save HPO trials table
    trials_df = pd.DataFrame(trial_rows)
    trials_csv = ART_DIR / f"hpo_trials_{target}.csv"
    trials_df.to_csv(trials_csv, index=False)
    print(f"Saved HPO trials -> {trials_csv}")

    # Save BEST TRIAL loss curve (from the HPO phase)
    if best_logs is not None:
        hpo_loss_df = pd.DataFrame(best_logs)
        hpo_loss_csv = ART_DIR / f"hpo_best_trial_loss_{target}.csv"
        hpo_loss_df.to_csv(hpo_loss_csv, index=False)
        print(f"Saved best HPO trial loss curve -> {hpo_loss_csv}")

    # === Retrain best config on FULL TRAIN (tiny dummy val just to reuse loop) ===
    dummy_Xva, dummy_yva = Xtr_np[:1], y_train_vec[:1]

    final = train_eval_one(
        Xtr_np, y_train_vec, dummy_Xva, dummy_yva,
        Xte_np, y_test_vec,
        in_dim=in_dim,
        hidden_dims=best_conf["hidden"],
        dropout=best_conf["dropout"],
        activation=best_conf["activation"],
        lr=best_conf["lr"],
        weight_decay=best_conf["weight_decay"],
        batch_size=best_conf["batch_size"],
        max_epochs=max(20, best["best_epoch"]),
        patience=12,
        clip_norm=1.0,
        verbose=False
    )
    final_metrics = final["metrics"]
    print(f"\n=== Final Test ({target}) ===")
    for k, v in final_metrics.items():
        print(f"{k:>12}: {v:.6f}" if isinstance(v, float) else f"{k:>12}: {v}")

    # Save artifacts
    feat_list = feat_list_global
    joblib.dump(feat_list, ART_DIR / f"features_{target}.pkl")
    torch.save(final["model"].state_dict(), ART_DIR / f"mlp_{target}.pth")
    with open(ART_DIR / f"best_config_{target}.json", "w") as f:
        json.dump(best_conf, f, indent=2)

    # Save predictions vs truth (final best model)
    pred_df = pd.DataFrame({"y_true": final["y_true"], "y_pred": final["y_pred"]})
    pred_csv = ART_DIR / f"pred_vs_true_{target}.csv"
    pred_df.to_csv(pred_csv, index=False)

    # Save FINAL retrain loss curve
    final_loss_df = pd.DataFrame(final["logs"])
    final_loss_csv = ART_DIR / f"final_loss_curve_{target}.csv"
    final_loss_df.to_csv(final_loss_csv, index=False)

    print(f"Saved: weights/features/config/predictions/loss-curves for target={target} in {ART_DIR}")



Target: score_gpt4o
  Trial 01 | val*=23.424320 | test RMSE=4.84360 | r=0.7080
    Tried: {'hidden': (512, 256, 128), 'dropout': 0.3, 'activation': 'relu', 'lr': 0.001, 'weight_decay': 1e-06, 'batch_size': 256}
    Best : {'hidden': (512, 256, 128), 'dropout': 0.3, 'activation': 'relu', 'lr': 0.001, 'weight_decay': 1e-06, 'batch_size': 256}

  Trial 02 | val*=23.520929 | test RMSE=4.83121 | r=0.7100
    Tried: {'hidden': (1024, 512, 256, 128), 'dropout': 0.1, 'activation': 'relu', 'lr': 0.001, 'weight_decay': 1e-06, 'batch_size': 128}
    Best : {'hidden': (512, 256, 128), 'dropout': 0.3, 'activation': 'relu', 'lr': 0.001, 'weight_decay': 1e-06, 'batch_size': 256}

  Trial 03 | val*=23.983646 | test RMSE=4.83576 | r=0.7120
    Tried: {'hidden': (1024, 1024, 128, 128, 128, 128), 'dropout': 0.2, 'activation': 'relu', 'lr': 0.0001, 'weight_decay': 0.0001, 'batch_size': 128}
    Best : {'hidden': (512, 256, 128), 'dropout': 0.3, 'activation': 'relu', 'lr': 0.001, 'weight_decay': 1e-06, 'b

In [19]:

# =========================
# Config
# =========================
ART_DIR = Path("mlp_llm_targets_artifacts")  # where HPO saved artifacts
ART_DIR.mkdir(exist_ok=True, parents=True)

# If you know the exact targets, list them; otherwise we can auto-detect from artifacts.
# Common case:
TARGETS = ["score_gpt4o", "score_gemini", "score_deepseek", "llm_mean_score"]

# Input data and participants
DATA_PATH  = "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/bn_synthetic_generation_df.csv"
PARTS_PATH = "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/bn_synthetic_generation_participants_kept.csv"

# Outputs
OUT_DIR = Path(".")
COMBINED_OUT = OUT_DIR / "bn_synthetic_with_ann_predictions_ALL.csv"
PAIR_OUT_TPL = "/users/PDS0345/rxa615/Comorbidity Score/synthetic_data/df_ann_distilled_{target}.csv"

# Optional: if you want to reproduce earlier filtering, set True and adjust below
APPLY_OPTIONAL_FILTERING = False
FLAG_COL = "R95-R99 Ill-defined and unknown causes of mortality"

# =========================
# Model builder (matches HPO training)
# =========================
def make_activation(name: str):
    name = str(name).lower()
    return {"relu": torch.nn.ReLU, "leakyrelu": torch.nn.LeakyReLU}[name]()

class MLP(torch.nn.Module):
    def __init__(self, in_dim, hidden_dims, dropout, activation="relu"):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden_dims:
            layers += [torch.nn.Linear(last, int(h)), make_activation(activation), torch.nn.Dropout(float(dropout))]
            last = int(h)
        layers += [torch.nn.Linear(last, 1)]
        self.net = torch.nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Load data + participant ids
# =========================
df = pd.read_csv(DATA_PATH, low_memory=False)
parts = pd.read_csv(PARTS_PATH)
if len(parts) != len(df):
    print("Warning: participants file length != data length; aligning by index.")
    parts = parts.reindex(df.index)
df["participant_id"] = parts["participant_id"].astype(str).values

# Optional filter to reproduce older mean-model pipeline (usually not needed if features are aligned)
if APPLY_OPTIONAL_FILTERING:
    # drop “R00-R99*” cols and "ethnicity_name" if present
    drop_cols = [c for c in df.columns if c.startswith("R00-R99")]
    if "ethnicity_name" in df.columns:
        drop_cols.append("ethnicity_name")
    df = df.drop(columns=drop_cols, errors="ignore").copy()
    if FLAG_COL in df.columns:
        before = len(df)
        df = df.loc[df[FLAG_COL] != 1].copy()
        after = len(df)
        if before != after:
            print(f"Dropped {before - after} rows with '{FLAG_COL} == 1'.")
        df.drop(columns=[FLAG_COL], inplace=True)

# We’ll build a combined predictions frame on the subset we actually predict
combined_out = df.copy()

# =========================
# Helper: predict for a single target
# =========================
def predict_one_target(target: str, base_df: pd.DataFrame):
    # Artifact paths
    weights_path  = ART_DIR / f"mlp_{target}.pth"
    features_path = ART_DIR / f"features_{target}.pkl"
    config_path   = ART_DIR / f"best_config_{target}.json"
    scaler_path   = ART_DIR / f"scaler_{target}.pkl"      # optional; may not exist

    # Sanity checks
    if not features_path.exists():
        raise FileNotFoundError(f"[{target}] Missing features file: {features_path}")
    if not config_path.exists():
        raise FileNotFoundError(f"[{target}] Missing best-config file: {config_path}")
    if not weights_path.exists():
        raise FileNotFoundError(f"[{target}] Missing weights: {weights_path}")

    # Load artifacts
    feature_list = joblib.load(features_path)
    with open(config_path, "r") as f:
        conf = json.load(f)

    # OPTIONAL scaler
    scaler = None
    if scaler_path.exists():
        scaler = joblib.load(scaler_path)
        print(f"[{target}] Using scaler: {scaler_path.name}")
    else:
        print(f"[{target}] No scaler found (proceeding without).")

    # Align columns: add missing=0, drop extras, reorder
    X_df = base_df.copy()
    # ensure participant_id exists for later merging
    if "participant_id" not in X_df.columns:
        raise RuntimeError(f"[{target}] 'participant_id' column not present after loading data.")

    missing = [c for c in feature_list if c not in X_df.columns]
    for c in missing:
        X_df[c] = 0
    extra = [c for c in X_df.columns if c not in feature_list]
    if extra:
        X_df = X_df.drop(columns=extra)
    X_df = X_df[feature_list]

    # Build model according to saved config
    hidden     = tuple(conf["hidden"]) if isinstance(conf["hidden"], (list, tuple)) else tuple(eval(str(conf["hidden"])))
    dropout    = float(conf.get("dropout", 0.2))
    activation = str(conf.get("activation", "relu"))
    in_dim     = len(feature_list)

    model = MLP(in_dim=in_dim, hidden_dims=hidden, dropout=dropout, activation=activation).to(device)

    # Load state dict
    state_dict = torch.load(weights_path, map_location=device)
    # Safety: check input size in first layer
    first_linear_key = [k for k in state_dict.keys() if k.endswith(".weight")][0]
    expected_in = state_dict[first_linear_key].shape[1]
    if expected_in != in_dim:
        raise RuntimeError(f"[{target}] Model expects {expected_in} features, but aligned frame has {in_dim}.")
    model.load_state_dict(state_dict)
    model.eval()

    # Prepare features (float32), apply scaler if present
    X_np = X_df.values.astype("float32")
    if scaler is not None:
        X_np = scaler.transform(X_np)
    X_t = torch.tensor(X_np, device=device)

    # Predict
    with torch.no_grad():
        y_pred = model(X_t).cpu().numpy().ravel()

    # Return a pair frame with participant_id and prediction
    pred_col = f"{target}_distilled"
    pair_df = pd.DataFrame({
        "participant_id": base_df.loc[X_df.index, "participant_id"].astype(str).values,
        pred_col: y_pred
    })

    return pair_df, pred_col

# =========================
# Run all targets
# =========================
all_pair_files = []
for tgt in TARGETS:
    print(f"\n=== Predicting target: {tgt} ===")
    pair_df, pred_col = predict_one_target(tgt, combined_out)
    # Merge onto combined_out (align by participant_id)
    combined_out = combined_out.merge(pair_df, on="participant_id", how="left")

    # Save pair file for this target
    pair_path = Path(PAIR_OUT_TPL.format(target=tgt))
    pair_df.to_csv(pair_path, index=False)
    all_pair_files.append(str(pair_path))
    print(f"Saved per-target pairs -> {pair_path}")

# Save combined file with all predictions
combined_out.to_csv(COMBINED_OUT, index=False)
print("\n== Summary ==")
print(f"Combined predictions: {COMBINED_OUT}")
for p in all_pair_files:
    print(f"Pair file: {p}")



=== Predicting target: score_gpt4o ===
[score_gpt4o] No scaler found (proceeding without).
Saved per-target pairs -> /users/PDS0345/rxa615/Comorbidity Score/synthetic_data/df_ann_distilled_score_gpt4o.csv

=== Predicting target: score_gemini ===
[score_gemini] No scaler found (proceeding without).
Saved per-target pairs -> /users/PDS0345/rxa615/Comorbidity Score/synthetic_data/df_ann_distilled_score_gemini.csv

=== Predicting target: score_deepseek ===
[score_deepseek] No scaler found (proceeding without).
Saved per-target pairs -> /users/PDS0345/rxa615/Comorbidity Score/synthetic_data/df_ann_distilled_score_deepseek.csv

=== Predicting target: llm_mean_score ===
[llm_mean_score] No scaler found (proceeding without).
Saved per-target pairs -> /users/PDS0345/rxa615/Comorbidity Score/synthetic_data/df_ann_distilled_llm_mean_score.csv

== Summary ==
Combined predictions: bn_synthetic_with_ann_predictions_ALL.csv
Pair file: /users/PDS0345/rxa615/Comorbidity Score/synthetic_data/df_ann_dis